# JAX features for numerical computing: an overview

This notebook is the entry point for the heat-equation, wave-equation, and N-body tutorials in this directory. It isolates the JAX ideas shared by all three so that the application notebooks can concentrate on numerical science.

JAX is best understood not as “NumPy on a GPU,” but as a system for applying program transformations to array-oriented Python functions. The important transformations here are compilation, structured control flow, vectorization, and automatic differentiation.

> JAX changes how a computation is expressed and executed. It does not select a stable discretization, validate a model, or guarantee that a GPU is faster.

## How to use this overview

**Audience.** Scientific Python users who know basic NumPy and want to understand the JAX programming model before studying larger examples. No machine-learning background is assumed.

**Prerequisites.** Required: Python functions, array shapes, indexing, and loops. Helpful but optional: derivatives and numerical time integration.

**Time.** Allow 30–40 minutes for the guided sections and 10–20 minutes for the exercises. The notebook is self-contained and should be run from top to bottom in a fresh kernel.

**Environment.** Use the `environment.yml` in this directory. Exercises and complete solutions are grouped near the end.

## Learning objectives

By the end, you should be able to:

1. explain why pure functions and immutable arrays enable JAX transformations;
2. choose `jit`, `fori_loop`, `scan`, `vmap`, or autodiff for a stated numerical task;
3. compose transformations around the same numerical function;
4. distinguish compilation time, asynchronous execution, and data-transfer time;
5. anticipate recompilation, precision, memory, and scaling problems; and
6. identify which application notebook provides deeper practice for each feature.

## Feature map

| JAX idea | What problem it addresses | Heat equation | Wave equation | N-body |
|---|---|:---:|:---:|:---:|
| `jax.numpy` arrays | NumPy-like array expressions on JAX backends | ✓ | ✓ | ✓ |
| immutable functional updates | transformable state without hidden mutation | ✓ | — | — |
| explicit PyTree state | several arrays move through one computation | — | two time levels | positions, velocities, acceleration |
| `jax.jit` | compile a complete numerical kernel | ✓ | ✓ | ✓ |
| static arguments and shapes | define the structure of a compiled program | step count | step count | step count and body count |
| `jax.lax.fori_loop` | iterate when only a final state is required | time stepping | final wave state | final particle state |
| `jax.lax.scan` | iterate while returning a selected history | — | wave history | trajectories |
| `jax.vmap` | map one function over an array axis | diffusivity ensemble | energy over time | diagnostics and orbit ensemble |
| `value_and_grad` | differentiate a scalar diagnostic | diffusivity mismatch | wave-speed amplitude | initial-speed sensitivity |
| `block_until_ready` | make accelerator timings wait for completion | ✓ | implicit validation sync | implicit validation sync |
| scientific validation | test the algorithm, not just execution | exact solution and convergence | convergence and energy | invariants and convergence |

A dash means the notebook does not emphasize the feature, not that JAX cannot use it there.

## 1. Setup: backend and precision are part of the experiment

The application notebooks enable 64-bit arithmetic for error and conservation studies. That differs from JAX's usual accelerator-oriented 32-bit default. Precision affects accuracy, memory traffic, and hardware throughput, so select it explicitly rather than inheriting it accidentally.

In [1]:
from time import perf_counter

import jax

jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import numpy as np

print(f"JAX version: {jax.__version__}")
print(f"Default backend: {jax.default_backend()}")
print(f"64-bit enabled: {jax.config.jax_enable_x64}")
for device in jax.devices():
    print(f"  {device}")

JAX version: 0.6.2
Default backend: cpu
64-bit enabled: True
  TFRT_CPU_0


A CPU result is valid. If an accelerator was expected but is absent, correct the environment before interpreting performance. Device discovery is a setup check, not a benchmark.

## 2. NumPy-like arrays, with an important difference

`jax.numpy` mirrors much of NumPy's array interface. JAX arrays are immutable: an expression cannot silently change an existing array. Use `.at[...].set(...)`, `.add(...)`, or related functional updates to create a new value.

Immutability makes data flow explicit. It does not necessarily imply a physical copy: compiled code may reuse storage when the transformation can prove that reuse is safe.

In [2]:
original = jnp.arange(5.0)
updated = original.at[2].set(-1.0)
indices = jnp.array([0, 2, 2])
accumulated = original.at[indices].add(jnp.array([10.0, 1.0, 2.0]))

print("original:   ", original)
print("updated:    ", updated)
print("accumulated:", accumulated)

original:    [0. 1. 2. 3. 4.]
updated:     [ 0.  1. -1.  3.  4.]
accumulated: [10.  1.  5.  3.  4.]


The repeated index in `.add(...)` is meaningful: both contributions to index 2 are accumulated. This is useful for assembly-like patterns, although competing updates and backend-specific implementations still deserve performance measurement.

## 3. Write transformation-friendly functions

A function intended for JAX transformation should receive changing values as arguments and return results explicitly. Avoid hidden mutation, data-dependent Python control flow, and conversion of traced values to ordinary Python numbers.

A periodic finite-difference operator is a representative pure array function:

In [3]:
def periodic_laplacian(values, dx):
    return (
        jnp.roll(values, 1) - 2.0 * values + jnp.roll(values, -1)
    ) / dx**2


sample = jnp.sin(2.0 * jnp.pi * jnp.arange(8) / 8)
print(periodic_laplacian(sample, dx=1.0 / 8))

[-1.42108547e-14 -2.65096680e+01 -3.74903320e+01 -2.65096680e+01
 -7.10542736e-15  2.65096680e+01  3.74903320e+01  2.65096680e+01]


This pattern appears in both PDE notebooks. The N-body force has the same contract but operates a larger collection of arrays. Purity is good software design in its own right; JAX makes it a practical requirement for reliable transformation.

## 4. `jit`: compile at a useful boundary

`jax.jit` traces array operations and compiles them for the selected backend. Compile an outer numerical operation—such as a complete solve—rather than decorating every small expression reflexively. A larger useful boundary gives the compiler more opportunity to optimize and reduces repeated Python dispatch.

The first call includes tracing and compilation. Later calls can reuse the executable when relevant shapes, dtypes, and static arguments match. Accelerator dispatch can be asynchronous, so `.block_until_ready()` is required for honest timing.

In [4]:
compiled_laplacian = jax.jit(periodic_laplacian)
large_sample = jnp.sin(
    2.0 * jnp.pi * jnp.arange(200_000, dtype=jnp.float64) / 200_000
)
large_dx = 1.0 / large_sample.size

start = perf_counter()
compiled_laplacian(large_sample, large_dx).block_until_ready()
first_call_time = perf_counter() - start

start = perf_counter()
compiled_result = compiled_laplacian(large_sample, large_dx)
compiled_result.block_until_ready()
warmed_call_time = perf_counter() - start

print(f"first call (compile + execute): {first_call_time:.4f} s")
print(f"warmed execution:              {warmed_call_time:.4f} s")

first call (compile + execute): 0.0256 s
warmed execution:              0.0009 s


These two numbers answer different questions. Neither is a fair CPU-versus-GPU comparison by itself. A representative benchmark must also define dtypes, transfer costs, warm-up, repetitions, and the problem sizes that matter to the application.

### Tracing, shapes, and static arguments

During tracing, JAX usually knows an array's shape and dtype but not its runtime values. Shapes often define storage and compiled structure. A new shape or dtype can therefore trigger recompilation. Python values that determine program structure—such as a loop length used by `scan`—are often marked static. Changing a static value also causes a new compilation.

`jax.make_jaxpr` shows the primitive array program produced by tracing; it is a diagnostic tool, not an API learners need to memorize.

In [5]:
traced_program = jax.make_jaxpr(periodic_laplacian)(jnp.ones(4), 0.25)
print(f"traced representation: {type(traced_program).__name__}")
print(f"primitive equations:   {len(traced_program.jaxpr.eqns)}")

traced representation: ClosedJaxpr
primitive equations:   8


## 5. Structured state: PyTrees

JAX transformations understand nested structures of arrays called PyTrees. Tuples, lists, dictionaries, and registered data classes can carry related state without packing physically different quantities into one artificial array.

The wave solver uses `(previous_displacement, current_displacement)`. The N-body solver uses `(positions, velocities, accelerations)`. The container structure is static; the array leaves contain changing numerical values.

In [6]:
state = {
    "position": jnp.array([1.0, 2.0]),
    "velocity": jnp.array([-0.5, 0.25]),
}

def double_leaf(array):
    return 2.0 * array


doubled_state = jax.tree_util.tree_map(double_leaf, state)
print(jax.tree_util.tree_structure(state))
print(doubled_state)

PyTreeDef({'position': *, 'velocity': *})
{'position': Array([2., 4.], dtype=float64), 'velocity': Array([-1. ,  0.5], dtype=float64)}


PyTrees organize state; they do not remove the need for clear meanings, compatible shapes, or documented units. A deeply nested state can become as difficult to understand as a monolithic array.

## 6. Compiled loops: `fori_loop` and `scan`

Long Python loops are a poor fit inside a compiled numerical solve: unrolling them can make tracing and compilation expensive, while leaving repeated operations outside `jit` causes repeated dispatch. `jax.lax` supplies control-flow primitives that remain inside the staged computation.

We use the simple recurrence $x^{n+1}=x^n(1-r\Delta t)$ as a miniature stand-in for time stepping.

In [7]:
def decay_final(initial, rate, dt, n_steps):
    """Use fori_loop when only the final state is needed."""
    def body(_, value):
        return value * (1.0 - rate * dt)

    return jax.lax.fori_loop(0, n_steps, body, initial)


def decay_history(initial, rate, dt, n_steps):
    """Use scan when selected output from every step is required."""
    def body(value, _):
        new_value = value * (1.0 - rate * dt)
        return new_value, new_value

    _, later_values = jax.lax.scan(
        body, initial, xs=None, length=n_steps
    )
    return jnp.concatenate((initial[None, :], later_values), axis=0)


decay_final_jit = jax.jit(decay_final, static_argnames=("n_steps",))
decay_history_jit = jax.jit(decay_history, static_argnames=("n_steps",))

initial_state = jnp.linspace(1.0, 2.0, 8)
rate = 0.7
dt = 0.01
n_steps = 100
final_state = decay_final_jit(initial_state, rate, dt, n_steps)
history = decay_history_jit(initial_state, rate, dt, n_steps)

print(f"final-state shape: {final_state.shape}")
print(f"history shape:     {history.shape}")
print(f"stored states:     {history.shape[0]}")

final-state shape: (8,)
history shape:     (101, 8)
stored states:     101


Both functions perform the same recurrence. `fori_loop` returns one state; `scan` returns 101 states and therefore consumes more memory. Store history because a diagnostic needs it, not because it is convenient during development. For large simulations, store selected diagnostics, subsampled states, or checkpoints.

## 7. `vmap`: map over an array axis

`jax.vmap` transforms a function written for one input into a function that maps over an array axis. It avoids manually threading a batch dimension through every expression. This is useful for parameter ensembles and for applying diagnostics across a stored trajectory.

Vectorization is not free: batched states coexist in memory, and a very large ensemble may need chunking or distribution.

In [8]:
rates = jnp.array([0.2, 0.7, 1.5])

def solve_one_rate(one_rate):
    return decay_final(initial_state, one_rate, dt, n_steps)


solve_rate_ensemble = jax.jit(jax.vmap(solve_one_rate))
ensemble_final_states = solve_rate_ensemble(rates)
print(f"rates shape:    {rates.shape}")
print(f"results shape:  {ensemble_final_states.shape}")
print("first component for each rate:", ensemble_final_states[:, 0])

rates shape:    (3,)
results shape:  (3, 8)
first component for each rate: [0.8185668  0.49536447 0.22060891]


The leading result axis corresponds to the leading `rates` axis. `vmap` changes scheduling and array structure, not the underlying numerical model.

## 8. Automatic differentiation

JAX can differentiate scalar diagnostics through array operations and fixed-length loops. This supports parameter sensitivities, inverse problems, design optimization, and adjoint-like calculations outside machine learning.

The derivative belongs to the **implemented discrete computation**. It is not automatically the derivative of the underlying continuous model. Convergence, conditioning, non-smooth branches, iterative stopping rules, and chaotic sensitivity still matter.

In [9]:
def final_mean_square(one_rate):
    final = decay_final(initial_state, one_rate, dt, n_steps)
    return jnp.mean(final**2)


objective_value, objective_gradient = jax.value_and_grad(final_mean_square)(rate)
print(f"objective: {float(objective_value):.6f}")
print(f"d(objective)/d(rate): {float(objective_gradient):.6f}")

objective: 0.578410
d(objective)/d(rate): -1.164974


The negative derivative is physically sensible: increasing the decay rate reduces the final squared magnitude. Sign and scale checks remain valuable even when the derivative was produced automatically.

## 9. Transformations compose

The main advantage is not any transformation in isolation. Pure functions let transformations be nested. The next cell differentiates one simulation, maps that value-and-gradient calculation over several rates, and compiles the complete ensemble.

In [10]:
ensemble_value_and_gradient = jax.jit(
    jax.vmap(jax.value_and_grad(final_mean_square))
)
ensemble_objectives, ensemble_gradients = ensemble_value_and_gradient(rates)

for one_rate, value, gradient in zip(
    np.asarray(rates),
    np.asarray(ensemble_objectives),
    np.asarray(ensemble_gradients),
):
    print(
        f"rate={one_rate:.1f}: objective={value:.6f}, gradient={gradient:.6f}"
    )

rate=0.2: objective=1.579407, gradient=-3.165145
rate=0.7: objective=0.578410, gradient=-1.164974
rate=1.5: objective=0.114718, gradient=-0.232930


A useful reading order for nested transformations is inside out:

1. `value_and_grad` turns one scalar-output solve into a value-and-gradient function;
2. `vmap` maps that function over rates;
3. `jit` compiles the batched calculation.

Not every mathematically possible nesting is efficient. Measure compile time, runtime, and memory for the actual workload.

## 10. Host/device boundaries

JAX arrays may live on an accelerator. Printing a value, converting it with `np.asarray`, or extracting a Python scalar with `float(...)` requires the host to observe the result and therefore synchronizes execution.

Keep numerical kernels in JAX. Convert to NumPy at explicit boundaries for plotting, file formats, or libraries that expect host arrays. Do not place `np.asarray` or `float` inside a function that must be transformed.

In [11]:
host_result = np.asarray(ensemble_final_states)
print("JAX result is jax.Array: ", isinstance(ensemble_final_states, jax.Array))
print("host result is ndarray:  ", isinstance(host_result, np.ndarray))
print(f"host array shape: {host_result.shape}")

JAX result is jax.Array:  True
host result is ndarray:   True
host array shape: (3, 8)


The transfer is correct and intentional here. Repeated transfers inside a time loop would force synchronization and can erase accelerator benefits.

## 11. Exercises

The overview ends here. These exercises test selection and diagnosis rather than recall of function signatures.

### Exercise 1 — choose the transformation (10 minutes)

For each task, choose the primary JAX mechanism and justify the choice. More than one transformation may eventually compose around the function.

1. Run one time integrator and retain only its final state.
2. Run the same integrator but return an energy value at every step.
3. Apply one solver independently to 50 parameter values.
4. Compute the sensitivity of a scalar final-time mismatch to one parameter.
5. Reduce Python dispatch and let the backend optimize a complete solve.

**Success criterion:** connect the required input/output contract to `fori_loop`, `scan`, `vmap`, autodiff, or `jit`, and mention one relevant cost.

<details>
<summary><strong>Solution</strong></summary>

1. Use `fori_loop`: it carries state without materializing a history. The step count may need to be static, and reverse-mode behaviour depends on the resulting loop form.
2. Use `scan`: it carries state and stacks selected per-step outputs. The cost is storage proportional to the returned history.
3. Use `vmap`: it maps the single-parameter solver over an array axis. The batch increases live memory and may need chunking.
4. Use `grad` or `value_and_grad`: the function must return a scalar and remain differentiable along the traced path. The result is a discrete sensitivity.
5. Use `jit` around the useful outer boundary. Compilation adds first-call latency, and new shapes, dtypes, or static values may compile again.

In practice, tasks 1–4 are commonly wrapped in `jit`.
</details>

### Exercise 2 — diagnose four misleading snippets (10 minutes)

Identify the problem and a defensible correction in each case:

```python
x[0] = 1.0                         # A
if x.sum() > 0: ...                # B, inside a jitted function
%time compiled_function(x)         # C, accelerator timing
compiled_function(jnp.ones(n))     # D, for many changing values of n
```

**Success criterion:** diagnose mutation, data-dependent Python control flow, asynchronous timing, and shape-driven recompilation without blaming the GPU or compiler generically.

<details>
<summary><strong>Solution</strong></summary>

- **A:** JAX arrays are immutable. Use `x.at[0].set(1.0)` and keep the returned array.
- **B:** Python needs a concrete Boolean while tracing sees an abstract value. For elementwise selection use `jnp.where`; for staged branches use `jax.lax.cond`. The correct choice depends on whether both branches may be evaluated.
- **C:** the measurement may stop after asynchronous dispatch. Warm the compiled function and time `compiled_function(x).block_until_ready()`. Decide separately whether compilation belongs in the measured workflow.
- **D:** changing array shapes commonly triggers compilation for each new shape. Reuse a small set of shapes, pad and mask when scientifically acceptable, or accept and account for compilation. Padding wastes computation and memory, so it is not automatically better.
</details>

In [12]:
def nonnegative_square(values):
    return jnp.where(values > 0.0, values**2, 0.0)


exercise_input = jnp.array([-2.0, 0.0, 3.0])
exercise_updated = exercise_input.at[0].set(1.0)
exercise_output = jax.jit(nonnegative_square)(exercise_updated)
print("functional update:", exercise_updated)
print("staged selection:", exercise_output)

functional update: [1. 0. 3.]
staged selection: [1. 0. 9.]


## 12. What JAX does not decide

The application notebooks deliberately keep these responsibilities visible:

- **Heat equation:** an explicit method still needs its diffusion stability limit and a convergence study.
- **Wave equation:** a stable scheme can still have numerical dispersion; the initialization and discrete energy matter.
- **N-body simulation:** integrator choice changes long-time energy behaviour, and direct all-pairs broadcasting remains $O(N^2)$ in work and memory.

Across all three, GPU execution can lose to CPU execution on small problems. Compilation cannot improve an unsuitable algorithm's asymptotic complexity, and automatic differentiation cannot certify that a sensitivity answers the scientific question intended.

## 13. Suggested learning path

1. **[Heat equation](heat_equation.ipynb):** start here for immutable arrays, `jit`, honest timing, `fori_loop`, a parameter ensemble, and a first differentiated solver.
2. **[Wave equation](wave_equation.ipynb):** continue with multi-array state, `scan`, conserved energy, numerical dispersion, and travelling waves.
3. **[N-body simulation](n_body.ipynb):** apply the same ideas to pairwise broadcasting, symplectic integration, PyTree state, multiple system sizes, batched initial conditions, and physical symmetries.

The order is pedagogical rather than mandatory. A reader interested mainly in particle methods can move directly to N-body after completing this overview.

## 14. Reproducibility checks

These small checks verify that the overview's examples agree with one another. They do not benchmark hardware or validate any of the larger scientific models.

In [13]:
np.testing.assert_allclose(np.asarray(original), np.arange(5.0))
np.testing.assert_allclose(
    np.asarray(history[-1]), np.asarray(final_state), rtol=1.0e-13
)
assert history.shape == (n_steps + 1, initial_state.size)
assert ensemble_final_states.shape == (rates.size, initial_state.size)
assert jnp.all(jnp.isfinite(ensemble_gradients))
assert objective_gradient < 0.0
np.testing.assert_allclose(
    np.asarray(exercise_output), np.array([1.0, 0.0, 9.0])
)
print("All JAX feature-overview checks passed.")

All JAX feature-overview checks passed.


## References

- [JAX: How to think in JAX](https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html)
- [JAX: Just-in-time compilation](https://docs.jax.dev/en/latest/jit-compilation.html)
- [JAX: Control-flow operators](https://docs.jax.dev/en/latest/control-flow.html)
- [JAX: Automatic vectorization](https://docs.jax.dev/en/latest/automatic-vectorization.html)
- [JAX: The Autodiff Cookbook](https://docs.jax.dev/en/latest/notebooks/autodiff_cookbook.html)
- [JAX: Benchmarking](https://docs.jax.dev/en/latest/benchmarking.html)
- [JAX: Default dtypes and the X64 flag](https://docs.jax.dev/en/latest/default_dtypes.html)